## Attention Mechanism


### Dot Product

In [12]:
import torch 
import torch.nn as nn
torch.manual_seed(123)
inputs = torch.rand(4,8) #(num_tokens,embedding_dim)
print(inputs.shape)

torch.Size([4, 8])


In [13]:
attn_scores = inputs @ inputs.T
print(attn_scores)
print(attn_scores.shape)

tensor([[1.6775, 1.2426, 1.7782, 1.3842],
        [1.2426, 1.4390, 1.9746, 1.8711],
        [1.7782, 1.9746, 3.7323, 3.3564],
        [1.3842, 1.8711, 3.3564, 3.5359]])
torch.Size([4, 4])


In [14]:
attn_weights = torch.softmax(attn_scores, dim=-1) 
print(attn_weights)

tensor([[0.2858, 0.1850, 0.3161, 0.2131],
        [0.1620, 0.1972, 0.3369, 0.3038],
        [0.0708, 0.0862, 0.4998, 0.3432],
        [0.0543, 0.0884, 0.3903, 0.4670]])


### Self Attention


In [4]:
class SelfAttention(nn.Module):
    def __init__(self, d_in,d_out,qkv_bias):
        super().__init__()
        self.w_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,x):
        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)
    
        attn_scores = queries @  keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=-1)
        context_vector = attn_weights @ values
        return context_vector

In [5]:
d_in = 8
d_out = 6   
x = torch.rand(10,8)
attn = SelfAttention(d_in,d_out,False)
context_vector = attn(x)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

tensor([[ 0.1229,  0.1945, -0.0422, -0.2549, -0.1012,  0.1217],
        [ 0.1220,  0.1912, -0.0429, -0.2546, -0.1003,  0.1217],
        [ 0.1219,  0.2023, -0.0409, -0.2562, -0.1030,  0.1216],
        [ 0.1233,  0.1945, -0.0429, -0.2541, -0.1006,  0.1212],
        [ 0.1223,  0.1882, -0.0442, -0.2528, -0.0982,  0.1213],
        [ 0.1224,  0.1910, -0.0428, -0.2544, -0.1000,  0.1220],
        [ 0.1218,  0.2004, -0.0428, -0.2532, -0.0993,  0.1202],
        [ 0.1225,  0.1932, -0.0430, -0.2538, -0.0996,  0.1213],
        [ 0.1225,  0.1927, -0.0422, -0.2552, -0.1012,  0.1221],
        [ 0.1217,  0.2007, -0.0419, -0.2551, -0.1016,  0.1209]],
       grad_fn=<MmBackward0>)
Context Vector Shape: torch.Size([10, 6])


### Causal Attention

In [6]:
import torch
import torch.nn as nn
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,droput,qkv_bias = False):
        super().__init__()
        self.d_out = d_out
        self.w_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.droput = nn.Dropout(droput)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens , d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
        attn_weights = torch.softmax(attn_scores/queries.shape[-1]**0.5,dim=-1)
        attn_weights = self.droput(attn_weights)

        context_vector = attn_weights @ values
        return context_vector
    

In [7]:
torch.manual_seed(123)

input = torch.rand(10,8)
batch = torch.stack((input,input),dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
ca = CausalAttention(d_in,d_out,context_length,0.0)
context_vector = ca(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")


torch.Size([2, 10, 8])
tensor([[[-0.0760, -0.5336,  0.1025, -0.1544,  0.0059, -0.2706],
         [-0.0761, -0.5327,  0.0994, -0.1532,  0.0049, -0.2711],
         [-0.0766, -0.5345,  0.0971, -0.1516,  0.0038, -0.2719],
         [-0.0765, -0.5339,  0.0944, -0.1502,  0.0024, -0.2707],
         [-0.0760, -0.5310,  0.0978, -0.1526,  0.0047, -0.2708],
         [-0.0765, -0.5338,  0.0997, -0.1530,  0.0049, -0.2712],
         [-0.0757, -0.5324,  0.0955, -0.1505,  0.0027, -0.2679],
         [-0.0755, -0.5330,  0.0963, -0.1512,  0.0032, -0.2694],
         [-0.0768, -0.5339,  0.1021, -0.1549,  0.0064, -0.2741],
         [-0.0763, -0.5327,  0.0941, -0.1507,  0.0028, -0.2720]],

        [[-0.0760, -0.5336,  0.1025, -0.1544,  0.0059, -0.2706],
         [-0.0761, -0.5327,  0.0994, -0.1532,  0.0049, -0.2711],
         [-0.0766, -0.5345,  0.0971, -0.1516,  0.0038, -0.2719],
         [-0.0765, -0.5339,  0.0944, -0.1502,  0.0024, -0.2707],
         [-0.0760, -0.5310,  0.0978, -0.1526,  0.0047, -0.2708],


### Multi Head Attention

In [8]:
import torch
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,droput,num_heads,qkv_bias = False):
        super().__init__()
        assert (d_out%num_heads==0) , "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out//num_heads

        self.w_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.w_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.w_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.out_proj = nn.Linear(d_out,d_out)
        self.dropout = nn.Dropout(droput)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))

    def forward(self,x):
        b, num_tokens, d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        #(b,num_tokens,d_in) -> (b,num_tokens,num_heads,head_dim)
        queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
        keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
        values = values.view(b,num_tokens,self.num_heads,self.head_dim)

        # Transpose: (b,num_tokens,num_heads,head_dim) -> (b,num_heads,num_tokens,head_dim)
        queries = queries.transpose(1,2)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = queries @ keys.transpose(2,3) 

        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]

        attn_scores.masked_fill(mask_bool,-torch.inf)

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim = -1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_scores @ values).transpose(1,2)
        
        context_vector = context_vector.contiguous().view(b,num_tokens,self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector
        

In [9]:
import torch
torch.manual_seed(123)

inputs = torch.rand(10,8)
batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

batch_size, context_length , d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in,d_out,context_length,0.0,num_heads=2)
context_vector = mha(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

torch.Size([2, 10, 8])
tensor([[[-0.0348,  0.0913, -0.3014,  0.4339,  0.3169,  0.1672],
         [-0.2907,  0.1028, -0.2143,  0.3709,  0.4754,  0.1350],
         [-0.2558,  0.1046, -0.2306,  0.4212,  0.4138,  0.1120],
         [-0.4640,  0.1197, -0.1597,  0.3605,  0.5647,  0.0796],
         [-0.6494,  0.0976, -0.0992,  0.2702,  0.6901,  0.1221],
         [-0.1541,  0.1457, -0.2584,  0.4040,  0.5015,  0.0889],
         [-0.7326,  0.0513, -0.0841,  0.2842,  0.5999,  0.1489],
         [-0.5212, -0.0450, -0.1591,  0.3875,  0.1911,  0.2648],
         [ 0.0269,  0.2226, -0.2960,  0.4035,  0.6054,  0.0342],
         [-0.5168,  0.0840, -0.1404,  0.3537,  0.4926,  0.1226]],

        [[-0.0348,  0.0913, -0.3014,  0.4339,  0.3169,  0.1672],
         [-0.2907,  0.1028, -0.2143,  0.3709,  0.4754,  0.1350],
         [-0.2558,  0.1046, -0.2306,  0.4212,  0.4138,  0.1120],
         [-0.4640,  0.1197, -0.1597,  0.3605,  0.5647,  0.0796],
         [-0.6494,  0.0976, -0.0992,  0.2702,  0.6901,  0.1221],


### KV Cache

In [10]:
import torch
import torch.nn as nn

class KVCache:
    def __init__(self):
        self.keys = None
        self.values = None

    def update(self, new_keys, new_values):
        if self.keys is None:
            self.keys = new_keys
            self.values = new_values
        else:
            self.keys = torch.cat([self.keys, new_keys], dim=2)   
            self.values = torch.cat([self.values, new_values], dim=2)  
        return self.keys, self.values
    
class CausalAttentionWithKVCache(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, kv_cache=None, start_pos=0):
        b, num_tokens, d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            keys, values = kv_cache.update(keys, values)

        total_seq_len = keys.shape[2]

        q_pos = torch.arange(start_pos, start_pos + num_tokens)
        k_pos = torch.arange(total_seq_len)

        mask = k_pos.unsqueeze(0) > q_pos.unsqueeze(1)  # shape (num_tokens, total_seq_len)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores / keys.shape[-1]**0.5
        attn_scores.masked_fill_(mask.unsqueeze(0).unsqueeze(0), -torch.inf)        
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector

In [11]:
import torch
torch.manual_seed(123)

# Test KV Cache
inputs = torch.rand(1, 5, 8)  # batch=1, seq_len=5, d_in=8
d_out = 6
num_heads = 2
attn_kv = CausalAttentionWithKVCache(8, d_out, 10, 0.0, num_heads)
kv_cache = KVCache()

# First forward pass
context1 = attn_kv(inputs, kv_cache, start_pos=0)
print("First pass context shape:", context1.shape)
print("Cache keys shape:", kv_cache.keys.shape)

# Second forward pass with new tokens
new_inputs = torch.rand(1, 3, 8)
context2 = attn_kv(new_inputs, kv_cache, start_pos=5)
print("Second pass context shape:", context2.shape)
print("Updated cache keys shape:", kv_cache.keys.shape)

First pass context shape: torch.Size([1, 5, 6])
Cache keys shape: torch.Size([1, 2, 5, 3])
Second pass context shape: torch.Size([1, 3, 6])
Updated cache keys shape: torch.Size([1, 2, 8, 3])


### Multi Query Attention

In [12]:
import torch
import torch.nn as nn

class MultiQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, self.head_dim, bias=qkv_bias)  
        self.w_value = nn.Linear(d_in, self.head_dim, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.w_query(x)  # (b, num_tokens, d_out)
        keys = self.w_key(x)       # (b, num_tokens, head_dim)
        values = self.w_value(x)   # (b, num_tokens, head_dim)

        # Reshape queries to (b, num_tokens, num_heads, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)

        keys = keys.unsqueeze(1).expand(-1, self.num_heads, -1, -1)  # (b, num_heads, num_tokens, head_dim)
        values = values.unsqueeze(1).expand(-1, self.num_heads, -1, -1)  # (b, num_heads, num_tokens, head_dim)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector

In [13]:
import torch
torch.manual_seed(123)

inputs = torch.rand(10, 8)
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
mqa = MultiQueryAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vector = mqa(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

torch.Size([2, 10, 8])
tensor([[[-0.0433, -0.5992, -0.0832, -0.1847, -0.0756, -0.1987],
         [-0.0694, -0.5787, -0.0796, -0.2136, -0.0832, -0.1850],
         [-0.1153, -0.6770, -0.1205, -0.1892, -0.1306, -0.2029],
         [-0.1271, -0.7031, -0.1224, -0.1615, -0.1378, -0.2171],
         [-0.1077, -0.6627, -0.1074, -0.1803, -0.1246, -0.2054],
         [-0.1196, -0.6626, -0.1094, -0.1864, -0.1271, -0.2032],
         [-0.0973, -0.6642, -0.1051, -0.1692, -0.1175, -0.2106],
         [-0.0779, -0.6674, -0.1056, -0.1572, -0.1080, -0.2159],
         [-0.0824, -0.6691, -0.1059, -0.1545, -0.1086, -0.2171],
         [-0.0833, -0.6501, -0.0968, -0.1579, -0.1001, -0.2147]],

        [[-0.0433, -0.5992, -0.0832, -0.1847, -0.0756, -0.1987],
         [-0.0694, -0.5787, -0.0796, -0.2136, -0.0832, -0.1850],
         [-0.1153, -0.6770, -0.1205, -0.1892, -0.1306, -0.2029],
         [-0.1271, -0.7031, -0.1224, -0.1615, -0.1378, -0.2171],
         [-0.1077, -0.6627, -0.1074, -0.1803, -0.1246, -0.2054],


### Grouped Query Attention

In [14]:
import torch
import torch.nn as nn

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, num_kv_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        assert (num_heads % num_kv_heads == 0), "num_heads must be divisible by num_kv_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_out // num_heads
        self.kv_head_dim = d_out // num_kv_heads  

        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, self.num_kv_heads * self.head_dim, bias=qkv_bias)
        self.w_value = nn.Linear(d_in, self.num_kv_heads * self.head_dim, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.w_query(x)  # (b, num_tokens, d_out)
        keys = self.w_key(x)       # (b, num_tokens, num_kv_heads * head_dim)
        values = self.w_value(x)   # (b, num_tokens, num_kv_heads * head_dim)

        # Reshape queries to (b, num_tokens, num_heads, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)

        # Reshape keys and values to (b, num_tokens, num_kv_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_kv_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_kv_heads, self.head_dim)

        # Transpose to (b, num_kv_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Repeat keys and values for each group of query heads
        queries_per_kv = self.num_heads // self.num_kv_heads
        keys = keys.repeat_interleave(queries_per_kv, dim=1)  # (b, num_heads, num_tokens, head_dim)
        values = values.repeat_interleave(queries_per_kv, dim=1)  # (b, num_heads, num_tokens, head_dim)

        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)
        return context_vector

In [15]:
import torch
torch.manual_seed(123)

inputs = torch.rand(10, 8)
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 8  
gqa = GroupedQueryAttention(d_in, d_out, context_length, 0.0, num_heads=4, num_kv_heads=2)
context_vector = gqa(batch)
print(context_vector)
print(f"Context Vector Shape: {context_vector.shape}")

torch.Size([2, 10, 8])
tensor([[[ 0.0298,  0.0218,  0.1682, -0.2304, -0.0879, -0.1733, -0.2138,
          -0.4307],
         [ 0.0660, -0.0099,  0.1553, -0.2558, -0.0869, -0.1476, -0.2690,
          -0.4187],
         [ 0.0953,  0.0011,  0.2098, -0.2634, -0.0901, -0.1232, -0.3437,
          -0.4455],
         [ 0.1327,  0.0027,  0.2051, -0.2952, -0.0996, -0.0868, -0.3943,
          -0.4454],
         [ 0.1077,  0.0013,  0.2367, -0.2684, -0.0896, -0.1087, -0.3790,
          -0.4534],
         [ 0.1046,  0.0100,  0.2319, -0.2661, -0.0920, -0.1107, -0.3680,
          -0.4546],
         [ 0.1044,  0.0185,  0.2415, -0.2704, -0.0946, -0.1098, -0.3737,
          -0.4620],
         [ 0.0976,  0.0177,  0.2384, -0.2674, -0.0933, -0.1151, -0.3626,
          -0.4597],
         [ 0.0876,  0.0161,  0.2717, -0.2554, -0.0866, -0.1090, -0.3651,
          -0.4594],
         [ 0.0874,  0.0181,  0.2582, -0.2624, -0.0894, -0.1091, -0.3569,
          -0.4561]],

        [[ 0.0298,  0.0218,  0.1682, -0.2304,

### Multi Head Latent Attention

In [1]:
import torch
import torch.nn as nn

class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model, num_heads, d_latent, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.d_latent = d_latent 
        self.w_q = nn.Linear(d_model, d_model)
        self.w_dkv = nn.Linear(d_model, d_latent)
        self.w_uk = nn.Linear(d_latent, d_model)
        self.w_uv = nn.Linear(d_latent, d_model)

        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(
            torch.ones(1, 1, 1024, 1024), diagonal=1).bool())

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        q = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        c_kv = self.w_dkv(x) # (batch, seq_len, d_latent)
        k = self.w_uk(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        v = self.w_uv(c_kv).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len], float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ v).transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model)

        output = self.w_o(context_vector)
        return output

In [2]:
d_model = 512
num_heads = 8
d_latent = 128  
batch_size = 4
seq_len = 64
mla = MultiHeadLatentAttention(d_model, num_heads, d_latent)
input = torch.randn(batch_size, seq_len, d_model)
output = mla(input)
print(f"Input shape: {input.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])
